In [0]:
dbutils.library.restartPython()

In [0]:
from common_utils.bronze import read_raw, bronze_ingestor
from common_utils.logging import get_logger
from datetime import date

import json

# ============================================================
# 0. LOGGER
# ============================================================
logger = get_logger("ds2b")

# ============================================================
# 1. CONFIG PATH
# ============================================================

dbutils.widgets.text("path","")
config_path = dbutils.widgets.get("path")

with open(config_path, "r") as f:
    config = json.load(f)

# ============================================================
# 0. LOGGER
# ============================================================
run_date = date.today().isoformat()
sources = config["sources"]

for source_name, config in sources.items():
    raw_path = f"/Volumes/employeedataplatform/bronze/raw_data/load_date={run_date}/{config['raw_folder']}/{source_name}.{config["file_format"]}"

    logger.info("[%s] reading from path %s", source_name, raw_path)
    df = read_raw(raw_path, file_format=config['file_format'], options=config['read_options'])

    rows = bronze_ingestor(df, mode=config['mode'], target_table=config['target_table'])
    logger.info("[%s] ingested %s rows into %s", source_name, rows, config['target_table'])

logger.info("bronze finished for all sources")